In [13]:
import pandas as pd
import joblib
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

DATA_DIR_PATH = "../data/"

In [2]:
INPUT_PATH = DATA_DIR_PATH + "preprocessed/normalized_dataset.csv"

df = pd.read_csv(INPUT_PATH)

df.head()

,link,date,title,content,keywords,category,normalized_title,normalized_content,normalized_keywords,normalized,tokens,filtered_tokens
0,https://entertainment.kompas.com/read/2026/07/...,2026-07-30 09:26:57,Deretan Aktor yang Pernah Perankan Spider-Man,"JAKARTA, KOMPAS.com- KehadiranSpider-Man: Bran...","peter parker, mcu, spider-man, tom holland",Hype,deretan aktor yang pernah perankan spider man,jakarta kehadiranspider man brand new dayyang ...,peter parker mcu spider man tom holland,deretan aktor yang pernah perankan spider man ...,"['deretan', 'aktor', 'yang', 'pernah', 'perank...","['deretan', 'aktor', 'perankan', 'spider', 'ma..."
1,https://www.kompas.com/hype/read/2026/08/04/16...,2026-08-04 16:51:09,Aktor Ryotaro Shimizu Ditemukan Meninggal Duni...,"KOMPAS.com- Aktor Ryotaro Shimizu, yang karier...","ryotaro shimizu, ryotaro shimizu meninggal",Hype,aktor ryotaro shimizu ditemukan meninggal duni...,aktor ryotaro shimizu yang karier hiburannya h...,ryotaro shimizu ryotaro shimizu meninggal,aktor ryotaro shimizu ditemukan meninggal duni...,"['aktor', 'ryotaro', 'shimizu', 'ditemukan', '...","['aktor', 'ryotaro', 'shimizu', 'ditemukan', '..."
2,https://entertainment.kompas.com/read/2026/07/...,2026-07-14 12:05:38,"Aktor Marvelous Mrs. Maisel, Josh Grisetti Men...","KOMPAS.com- Josh Grisetti, aktor yang dikenal ...","josh grisetti, josh grisetti meninggal, film j...",Hype,aktor marvelous mrs maisel josh grisetti menin...,josh grisetti aktor yang dikenal karena perann...,josh grisetti josh grisetti meninggal film jos...,aktor marvelous mrs maisel josh grisetti menin...,"['aktor', 'marvelous', 'mrs', 'maisel', 'josh'...","['aktor', 'marvelous', 'mrs', 'maisel', 'josh'..."
3,https://www.kompas.com/hype/read/2026/07/28/19...,2026-07-28 19:42:53,"Aktor Bittersweet Destiny, Wang Kai Ditemukan ...","KOMPAS.com- Aktor Taiwan Wang Kai, yang terken...","wang kaimu, wang kaimu drama, wang kai meninggal",Hype,aktor bittersweet destiny wang kai ditemukan m...,aktor taiwan wang kai yang terkenal karena per...,wang kaimu wang kaimu drama wang kai meninggal,aktor bittersweet destiny wang kai ditemukan m...,"['aktor', 'bittersweet', 'destiny', 'wang', 'k...","['aktor', 'bittersweet', 'destiny', 'wang', 'k..."
4,https://www.kompas.com/hype/read/2026/08/04/11...,2026-08-04 11:52:54,"Aktor Thailand, Jorjay Phumhirun Mendadak Meni...",KOMPAS.com- Aktor muda Thailand Jorjay Phumhir...,"jorjay phumhirun meninggal, jorjay phumhirun",Hype,aktor thailand jorjay phumhirun mendadak menin...,aktor muda thailand jorjay phumhirun mendadak ...,jorjay phumhirun meninggal jorjay phumhirun,aktor thailand jorjay phumhirun mendadak menin...,"['aktor', 'thailand', 'jorjay', 'phumhirun', '...","['aktor', 'thailand', 'jorjay', 'phumhirun', '..."


In [3]:
# Weighting filtered_tokens column

vectorizer = TfidfVectorizer()

matrix = vectorizer.fit_transform(df["filtered_tokens"])

print("TF-IDF Matrix Shape:", matrix.shape)

TF-IDF Matrix Shape: (8926, 95597)


In [9]:
n_rows, n_cols = matrix.shape
total_cells = n_rows * n_cols
n_nonzero = int(matrix.getnnz())
density_percent = 100 * (n_nonzero / total_cells) if total_cells else 0
sparsity_percent = 100 * (1 - (n_nonzero / total_cells)) if total_cells else 0

matrix_summary = pd.DataFrame({
    "metric": [
        "documents (rows)",
        "terms (columns)",
        "non-zero TF-IDF values",
        "total possible cells",
        "density (%)",
        "sparsity (%)",
    ],
    "value": [
        f"{n_rows:,}",
        f"{n_cols:,}",
        f"{n_nonzero:,}",
        f"{total_cells:,}",
        f"{density_percent:.2f}%",
        f"{sparsity_percent:.2f}%",
    ],
})

display(matrix_summary)

,metric,value
0,documents (rows),"8,926"
1,terms (columns),"95,597"
2,non-zero TF-IDF values,"1,511,438"
3,total possible cells,"853,298,822"
4,density (%),0.18%
5,sparsity (%),99.82%


In [11]:
# Ambil nama fitur sesuai urutan kolom pada matriks TF-IDF
feature_names = vectorizer.get_feature_names_out()

vocabulary_output = pd.DataFrame({
    "feature_index": np.arange(len(feature_names), dtype=int),
    "term": feature_names,
    "idf": vectorizer.idf_,
})

print("20 term dengan IDF tertinggi:")
display(
    vocabulary_output
    .nlargest(20, "idf")
    .sort_values("idf", ascending=False)
)

print("20 term dengan IDF terendah:")
display(
    vocabulary_output
    .nsmallest(20, "idf")
    .sort_values("idf", ascending=True)
)


20 term dengan IDF tertinggi:


,feature_index,term,idf
1,1,aabidah,9.403688
2,2,aad,9.403688
3,3,aadait,9.403688
5,5,aafaanii,9.403688
6,6,aafait,9.403688
7,7,aafinii,9.403688
8,8,aafiyatika,9.403688
9,9,aagesen,9.403688
10,10,aags,9.403688
11,11,aajali,9.403688


20 term dengan IDF terendah:


,feature_index,term,idf
6142,6142,baca,1.051370
34452,34452,jakarta,1.780047
48652,48652,masyarakat,1.879127
76216,76216,salah,1.908980
32522,32522,indonesia,1.916235
50896,50896,memiliki,1.970318
64303,64303,pemerintah,1.992434
42576,42576,kondisi,2.038508
61067,61067,orang,2.068381
71238,71238,proses,2.075579


In [14]:
# Save the vectorize weight

OUTPUT_VECTORIZER_PATH = DATA_DIR_PATH + "weighted/vectorizer.joblib"
OUTPUT_MATRIX_PATH = DATA_DIR_PATH + "weighted/matrix.npz"

save_npz(OUTPUT_MATRIX_PATH, matrix, compressed=True)
joblib.dump(vectorizer, OUTPUT_VECTORIZER_PATH)

print(f"TF-IDF Vectorizer saved to {OUTPUT_VECTORIZER_PATH}")

TF-IDF Vectorizer saved to ../data/weighted/vectorizer.joblib
